Now that we have shown that the k-fold models generalize better to the test data than a random baseline, we will generate a new model with the best hyperparameters identified, but trained on the training + validation data. This is simply to have one coherent model for additional downstream assessments. It also should give the model better power.

In [8]:
import os

import pandas as pd
import scanpy as sc

import torch
import torch.nn as nn
import numpy as np

import sys
sys.path.insert(1, '../.')
from Kang_utils import get_best_hyperparams

In [3]:
import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import io
from scLEMBAS.model.scl import SignalingModel
from scLEMBAS.model.train import TrainSC


In [4]:
seed = 888
device = "cuda" if torch.cuda.is_available() else "cpu"

data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'
models_path = os.path.join(data_path, 'processed', 'models')

In [5]:
adata = sc.read_h5ad(os.path.join(data_path, 'processed', 'kang_expr_scored.h5ad'))
sn_ppis = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_sn_ppis.csv'), index_col = 0)

tf_adata = io.read_tfad(os.path.join(data_path, 'processed', 'Kang_tf_activity.h5ad'))
tf_adata.obs['condition'] = tf_adata.obs['stim'].astype(str) + '^' + tf_adata.obs['seurat_annotations'].astype(str)

# ensures correct order of test data
# note, this already saved in order, matching the mod.y_out columns
tf_adata = tf_adata[:, sorted(tf_adata.var_names)] 

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [ ]:
test_cells = open(os.path.join(data_path, 'processed', 'data_split_barcodes', 'kang_test.txt')).read().splitlines()
train_cells = set(tf_adata.obs_names).difference(test_cells)

In [12]:
res = pd.read_csv(os.path.join(data_path, 'processed', 'Kang_k_fold_validation_results.csv'), index_col = 0)
best_emd_mean, best_hyperparams, best_emd = get_best_hyperparams(res)

trainers_best = io.read_pickled_object(os.path.join(data_path, 'processed', 'Kang_trainers_k.pickle'))


Select a hyper-parameter tuning model that uses the most common KL regularization value:

In [31]:
kl_reg = best_emd.KL_regularization.value_counts().idxmax()

kl_og = np.nan
for k, trainer_k in trainers_best.items():
    if kl_og != kl_reg:
        print(k)
        kl_og = trainer_k.hyper_params['vae_scaling_KL']
        mod_actual = trainer_k.mod
    else:
        break

0


Build and train the full model:

In [ ]:
mod_full = SignalingModel(net = sn_ppis,
                     X_in = mod_actual.X_in.copy(),
                     y_out = mod_actual.y_out.copy(), 
                     expr = mod_actual.expr.copy(), 
                     covariates = mod_actual.signaling_network.covariates.copy(),
                     categorical_covariate_keys = mod_actual.signaling_network.covariates.columns.tolist(),
                     projection_amplitude_in = mod_actual.input_layer.projection_amplitude, 
                     projection_amplitude_out = mod_actual.projection_amplitude_out,
                     weight_label = weight_label, source_label = source_label, target_label = target_label,
                     bionet_params = mod_actual.signaling_network.bionet_params.copy(), 
                     dtype = torch.float32, device = device, seed = seed)
mod_full.input_layer.weights.requires_grad = False # don't learn scaling factors for the ligand input concentrations
mod_full.signaling_network.prescale_weights(target_radius = mod_actual.signaling_network.bionet_params['spectral_target']) # spectral radius

hyper_params = trainer_k.hyper_params
hyper_params['validation_batch_size'] = np.nan
hyper_params['test_batch_size'] = len(test_cells)
trainer = TrainSC(mod = mod_full,
                  prediction_optimizer = torch.optim.Adam,
                  prediction_loss_fn = SamplesLoss("sinkhorn", p=2, blur=0.05).to(device), #torch.nn.MSELoss(reduction='mean'),
                  discriminator_params = trainer_k.discriminator['params'].copy(),
                  hyper_params = hyper_params.copy(),
              train_split = {'train': train_cells, 'test': test_cells, 'validation': None},
              train_seed = seed,
              track_test = False,
              track_validation = False)

mod_full = trainer.train_model(verbose = False)
io.write_pickled_object(trainer, os.path.join(data_path, 'processed', 'Kang_fullbest_trainer.pickle'))
